# 🧠 AETHER Autonomous Fine-Tuning on Kaggle (Dual T4 GPU)
This notebook fine-tunes **AETHER** on Kaggle's free **NVIDIA Tesla T4 GPUs** using **Unsloth (QLoRA)**, and exports the final model to **GGUF (Q4_K_M)** for running locally on your laptop or external drive.

In [ ]:
# 1. Install Unsloth and Required Libraries (Optimized for Kaggle T4)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Load Base Model in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✓ Model and LoRA adapters successfully initialized!")

In [ ]:
# 3. Load AETHER Curated Dataset
# If running via Kaggle Dataset, path is: /kaggle/input/aether-training-data/aether_train_dataset.jsonl
# Or create inline if uploaded directly to current directory
from datasets import load_dataset
import os

dataset_path = "aether_train_dataset.jsonl"
if not os.path.exists(dataset_path):
    # Fallback to search in Kaggle input
    for root, dirs, files in os.walk("/kaggle/input"):
        if "aether_train_dataset.jsonl" in files:
            dataset_path = os.path.join(root, "aether_train_dataset.jsonl")
            break

print(f"Using dataset: {dataset_path}")
dataset = load_dataset("json", data_files=dataset_path, split="train")

def format_prompts(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched = True)
print(f"✓ Processed {len(dataset)} training examples.")

In [ ]:
# 4. Fine-Tune AETHER with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting training on GPU...")
trainer.train()
print("✓ Training complete!")

In [ ]:
# 5. Save & Quantize Directly to GGUF
print("Exporting GGUF weights...")
model.save_pretrained_gguf("aether_trained_model", tokenizer, quantization_method = "q4_k_m")
print("✓ GGUF model saved in 'aether_trained_model/unsloth.Q4_K_M.gguf'. You can download this from Kaggle outputs!")